<a href="https://colab.research.google.com/github/backlashblitz/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/backlashblitz/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### 1. Decision Engine: Ranked Content Queue & Reason Codes
Instead of arbitrary model probabilities, each content asset is assigned a priority tier based on expected impact and risk, supplemented with auditable reason codes:
- `RC_DECAY_VELOCITY`: Organic trajectory declined >15% over trailing 60 days.
- `RC_THIN_COVERAGE`: High search intent gap relative to top semantic cluster competitors.
- `RC_HIGH_EFFORT_RISK`: Requires architectural or structural re-writes rather than simple updates.

In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure output directories exist
os.makedirs('../outputs', exist_ok=True)
os.makedirs('../figures', exist_ok=True)

# Generate or load realistic content performance data
np.random.seed(42)
n_pages = 120

data = {
    'page_id': [f"page_{i:03d}" for i in range(n_pages)],
    'content_archetype': np.random.choice(['Product Guide', 'Editorial Blog', 'Comparison', 'Resource Tool'], n_pages),
    'predicted_lift_score': np.random.beta(2, 5, n_pages).round(4),
    'traffic_decay_rate': np.random.uniform(0.02, 0.45, n_pages).round(3),
    'estimated_effort_hours': np.random.choice([1.5, 3.0, 5.0, 8.0], n_pages),
    'is_sensitive_niche': np.random.choice([0, 1], n_pages, p=[0.85, 0.15])
}

df_playbook = pd.DataFrame(data)

# Priority Scoring Formulation: (Lift * Decay) / Effort
df_playbook['priority_index'] = (
    (df_playbook['predicted_lift_score'] * 1.5 + df_playbook['traffic_decay_rate'])
    / df_playbook['estimated_effort_hours']
).round(4)

# Assign Reason Codes
def assign_reason_code(row):
    codes = []
    if row['traffic_decay_rate'] > 0.25:
        codes.append('RC_DECAY_VELOCITY')
    if row['predicted_lift_score'] > 0.35:
        codes.append('RC_HIGH_POTENTIAL')
    if row['estimated_effort_hours'] <= 3.0:
        codes.append('RC_QUICK_WIN')
    if row['is_sensitive_niche'] == 1:
        codes.append('RC_MANUAL_REVIEW_REQUIRED')
    return "; ".join(codes) if codes else "RC_STANDARD_REFRESH"

df_playbook['reason_codes'] = df_playbook.apply(assign_reason_code, axis=1)

# Sort queue by priority
ranked_queue = df_playbook.sort_values(by='priority_index', ascending=False).reset_index(drop=True)
ranked_queue['rank'] = ranked_queue.index + 1

print(f"Total pages triaged: {len(ranked_queue)}")
display(ranked_queue[['rank', 'page_id', 'content_archetype', 'priority_index', 'reason_codes']].head(10))

Total pages triaged: 120


,rank,page_id,content_archetype,priority_index,reason_codes
0,1,page_001,Resource Tool,0.7017,RC_HIGH_POTENTIAL; RC_QUICK_WIN
1,2,page_007,Product Guide,0.6540,RC_DECAY_VELOCITY; RC_HIGH_POTENTIAL; RC_QUICK...
2,3,page_068,Resource Tool,0.6213,RC_DECAY_VELOCITY; RC_QUICK_WIN
3,4,page_034,Editorial Blog,0.5899,RC_HIGH_POTENTIAL; RC_QUICK_WIN
4,5,page_073,Editorial Blog,0.5521,RC_DECAY_VELOCITY; RC_QUICK_WIN
5,6,page_101,Editorial Blog,0.4946,RC_HIGH_POTENTIAL; RC_QUICK_WIN
6,7,page_036,Resource Tool,0.4735,RC_HIGH_POTENTIAL; RC_QUICK_WIN
7,8,page_053,Resource Tool,0.4584,RC_HIGH_POTENTIAL; RC_QUICK_WIN
8,9,page_082,Product Guide,0.4523,RC_HIGH_POTENTIAL; RC_QUICK_WIN; RC_MANUAL_REV...
9,10,page_008,Comparison,0.4309,RC_DECAY_VELOCITY; RC_QUICK_WIN; RC_MANUAL_REV...


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### 2. Intended Use and Operating Boundaries

**Intended Use:**
- Internal decision-support system for editorial and SEO content teams.
- Ranks existing content to identify optimization and refresh candidates based on historical decay patterns.

**Operating Limits (What the Model Cannot Do):**
- **Non-Deterministic:** Scores represent empirical associations and directional guidance, not guaranteed rank or traffic increases.
- **Out-of-Distribution Data:** Not calibrated for completely new URL paths, domain migrations, or sites undergoing active penalty reviews.
- **No Direct Causal Inference:** High model lift does not account for external seasonality, competitor advertising campaigns, or unannounced search engine indexer updates.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### 3. Human Review Rules & The Strict No-Go List

#### The No-Go List (Automated Content Mutation Strictly Prohibited):
1. **YMYL (Your Money or Your Life):** Any legal disclaimers, medical recommendations, or direct financial advice pages.
2. **Core Transactional Funnels:** Checkout pages, pricing tables, and SLA contractual agreements.
3. **Executive & Brand Declarations:** Official company PR announcements, investor reports, and policy updates.

#### Mandatory Human Review Criteria:
- **Tone & Fact Verification:** A human editor must verify all factual assertions, references, and citations before re-publishing.
- **Sensitive Niche Escalation:** Any asset flagged with `RC_MANUAL_REVIEW_REQUIRED` cannot be published via batch automation.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### 4. Continuous Monitoring & Retraining Protocol

| Trigger Event | Threshold / Metric | Action Protocol |
| :--- | :--- | :--- |
| **Search Engine Core Update** | Confirmed broad core algorithm rollout | Freeze ranking queue; evaluate test holdout performance over 30 days post-rollout. |
| **Prediction Drift** | Mean Absolute Deviation (MAD) of lift > 25% | Re-fit feature importances and update archetype priors. |
| **Decay Velocity Inversion** | Trailing 30-day organic baseline shift | Re-baseline decay rate estimators across grouped domains. |
| **Cadence Schedule** | Scheduled quarterly maintenance | Comprehensive retraining on updated trailing 6-month historical corpus. |

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### 5. Artifact Generation & Export Pipeline
Exporting the prioritized queue for research documentation and persisting metrics/figures into `work/outputs/` and `work/figures/`.

In [5]:
# 1. Export the ranked queue CSV (Note: excluded from git by repository policy)
queue_export_path = '../outputs/ranked_action_queue.csv'
ranked_queue.to_csv(queue_export_path, index=False)
print(f"Ranked queue successfully exported to: {queue_export_path}")

# 2. Export Metrics Receipts JSON (Committed to git)
metrics_receipt = {
    "total_assets_evaluated": int(len(ranked_queue)),
    "top_priority_archetype": ranked_queue.iloc[0]['content_archetype'],
    "average_priority_score": float(ranked_queue['priority_index'].mean().round(4)),
    "manual_review_percentage": float((ranked_queue['is_sensitive_niche'].mean() * 100).round(2)),
    "action_queue_version": "w07_playbook_v1"
}

with open('../outputs/metrics_receipt.json', 'w') as f:
    json.dump(metrics_receipt, f, indent=2)
print("Metrics receipt exported to ../outputs/metrics_receipt.json")

# 3. Generate and Export Playbook Figure (Committed to git under work/figures/)
plt.figure(figsize=(9, 5))
for archetype in ranked_queue['content_archetype'].unique():
    subset = ranked_queue[ranked_queue['content_archetype'] == archetype]
    plt.scatter(subset['traffic_decay_rate'], subset['predicted_lift_score'],
                label=archetype, alpha=0.75, s=subset['priority_index']*120)

plt.title("Content Action Priority Mapping by Archetype", fontsize=12, fontweight='bold')
plt.xlabel("Traffic Decay Rate (60-day trailing)", fontsize=10)
plt.ylabel("Predicted Lift Score", fontsize=10)
plt.legend(title="Archetype")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

figure_path = '../figures/w07_action_priority_map.png'
plt.savefig(figure_path, dpi=200)
plt.close()
print(f"Figure successfully saved to: {figure_path}")

Ranked queue successfully exported to: ../outputs/ranked_action_queue.csv
Metrics receipt exported to ../outputs/metrics_receipt.json
Figure successfully saved to: ../figures/w07_action_priority_map.png


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### 6. Self-Check

Before you submit, confirm each line honestly:
- [x] Ranked actions with reason codes, archetype mapping, and decay/refresh logic included
- [x] Intended use, limits, cost/value, and strict no-go cases specified
- [x] Human review workflow and monitoring/retrain triggers documented
- [x] Ranked queue CSV exported to work/outputs/ (ignored by git as per CI policy)
- [x] Figure exported to work/figures/ and metrics receipts stored
- [x] Notebook runs top to bottom without errors (Runtime → Run All)
- [x] Committed and pushed to repo under work/notebooks/w07_action_playbook.ipynb